In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Let's see how embeddings work on a few examples.


In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. We can't read off what any one of them means. But two vectors with similar values point to texts about similar things.

Encode our document:

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

Next, we compare the query against the document using dot product:

In [4]:
v1.dot(dv)

np.float32(0.32332397)

We get 0.32.

Now we try an unrelated query:

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730438)

And we get 0.01.

The first score for q1 vs d (0.32) is higher, so that query is more similar to the document about registration. The second score for q2 vs d sits near 0, because installing Docker has nothing to do with registration. A score near 0 means the two vectors are about as different as they can be.

That's the whole idea behind vector search: similar texts get similar vectors, and a dot product tells us how similar.

**Cosine similarity**

The all-MiniLM-L6-v2 model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. That's why the model documentation says it "uses cosine similarity."

Cosine similarity measures the angle between two vectors, ignoring their length:

- 1.0 = same direction (similar)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite direction (opposite meaning)

Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):

cos(0) = 1 - vectors point in the same direction
cos(90) = 0 - vectors are perpendicular
cos(180) = -1 - vectors point in opposite directions

Because our vectors are normalized, the dot product gives us cosine similarity directly. This is why we can use v1.dot(dv) to compare texts.

In practice, we rarely get cosine similarity below 0. The embedding model maps text to a region of the vector space where most vectors have positive components. There's no concept of "opposite meaning" that maps to a vector pointing the other way.


### Embedding Our Dataset

In the previous lesson we saw how embeddings work on a couple of examples. Now we apply them to the whole FAQ dataset.

**Loading the data**

In module 1 we created ingest.py for loading the FAQ data.

Download it into your project:
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

We use it here:

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

**Generating embeddings**

Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

Build one text per document:

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

First we import tqdm so we can watch the progress:

In [11]:
from tqdm.auto import tqdm

Next we chunk the dataset into batches of 50 and encode each batch:

In [12]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

We end up with 1350 vectors. On a GPU this is fast. Most of us run on Codespaces without a GPU, so it takes a bit, but it's a one-off.

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [13]:
import numpy as np
X = np.array(vectors)

Calling X.shape returns (1208, 384) - number of documents vs number of dimensions.

### Vector Search manually

In the previous lesson we embedded our FAQ dataset into a matrix X with 1208 document vectors. Here we see how vector search works under the hood.

**Scoring documents**

We have a matrix X with all document embeddings. We take a query, compare it against every document, and keep the most similar ones.

When a query comes in, we embed it:

In [12]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

Next, we compute the dot product against all documents:

In [13]:
scores = X.dot(v_query)

This is matrix-vector multiplication. Each element i of scores is the cosine similarity between document i (row i of X) and v_query.

We could compute the same thing with a for loop:

In [14]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

But X.dot(v_query) is much faster. Numpy runs optimized C code instead of looping in Python, so the matrix version is hard to beat. The outcome is the same either way: one score per document.

**Best match**

The highest score is the most similar document:


In [15]:
idx = np.argmax(scores)
idx, scores[idx]


(np.int64(2), np.float32(0.76294106))

This returns document 553 with score 0.76.

The index and score may differ for you. Our FAQ is a living document, so we add and remove entries over time.

Let's see which document it is:

In [16]:
documents[idx]
#We see:

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

**Top 5 results**


Usually we want more than the single best match, so let's pull the top 5.

np.argsort sorts from lowest to highest, so the last 5 are the top ones:

In [17]:
top5 = np.argsort(scores)[-5:]

They come out smallest-first, so we reverse them to get the highest first:

In [18]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

Now we can read off the top 5 scores:

In [33]:
for i in range (top5.size):
    print(documents[top5[i]])

{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has alr

There's a shorter trick I usually reach for. We negate the scores first, so the largest becomes the smallest. Then argsort puts the best matches at the front.
Here it is in one line:

In [34]:
#it is not working
top5 = np.argsort(-scores)[:5]

TypeError: bad operand type for unary -: 'list'


It looks cryptic the first time you see it. But it's a common way to turn a min-sort into a max-sort.

Let's read off the actual documents:

In [35]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'


This is vector search in its simplest form. We embed the query, compute dot products against all documents, and return the highest-scoring ones.

We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. We send all 5 to the LLM and let it combine them.

The number 5 is a starting point, picked on gut feeling. Later, when we evaluate search quality, we can test whether 3 or 10 works better for our data.

Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. That's what we turn to next.

### Vector Search with minsearch

In the previous section we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. 

Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.


We'll use minsearch, the small in-memory search library we already used in module 1 for text search. 

It has a VectorSearch class for vector search.


Both classes share the same API:

- fit to index data
- search to query
- filter_dict in search to filter by keyword
- It's the simplest way to get started with vector search.

**Creating the index**

We already have our documents and vectors from the previous section.

Index them:

In [14]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

We pass the numpy array X with all embeddings and the list of documents as payload. The keyword_fields parameter works the same as in the text Index, so we can filter by course later.

**Searching**

Let's search for a question:

In [15]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.

Look at the top result:

In [16]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

**Filtering by course**

Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

Pass a filter_dict:

In [17]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

### RAG with Vector Search

In module 1, we built a RAG pipeline with three steps:
```
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

The search step used keyword search. Now we swap in vector search. Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.

**Using RAGBase**

In module 1 we put all the RAG logic into a RAGBase helper class. It has search, build_prompt, and llm methods, so we only need to override search.

Download rag_helper.py (and ingest.py if you didn't get it earlier) into your project:

- wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
- wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

First, create the OpenAI client:

In [22]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Next, download and index the data:

In [23]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Then use the RAGBase class:

In [24]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

Ask it a question:

This still uses keyword search. Text search isn't bad here, so the answer may already look right. Next we replace search with vector search.

In [25]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

We already have:

- All the indexed documents documents
- The embeddings matrix X with all these documents
- The vector search engine vindex
- We can't pass vindex to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass - RAGBase and override search to encode the query before searching.

The subclass overrides search:

In [26]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

The __init__ method adds one extra argument, embedder, for the sentence transformer. Inside search we use it to turn the query into a vector. 

Then we query vindex with that vector instead of the raw text. Everything else is inherited from RAGBase.

**Using it**

Let's init it:

In [27]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

Try it with different queries:

In [28]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still open.'

The answers should be close to what we got with keyword search, but vector search handles rephrased questions better. The swap was trivial because RAG has three clear steps. The same trick lets us change the LLM provider later by overriding just the llm step.

### Vector Search with sqlitesearch

In the previous section we used minsearch for vector search.

It works, but it has three problems:

* It rebuilds the index on every startup
* It keeps everything in memory
* It searches by brute force

With text search we never felt these. Indexing was fast because we didn't embed anything. With vector search, indexing runs a neural network over every document, so it takes a minute on our dataset. Keeping everything in memory is fine here, but a larger dataset would need too much space.

The third problem is brute-force search. For every query we compare the query vector against every single document. With 1,000 documents this is fine, probably even faster than anything smarter. But as the dataset grows past 10,000 or so, it gets slow, and we'll want an approximate method instead.

What we've done so far is exact **nearest neighbor** (NN) search. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays for that by touching everything.

**Approximate nearest neighbor (ANN)** search takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.

- NN (exact):    compare query against ALL documents -> top 5
- ANN (approx):  narrow down to a region -> compare within region -> top 5

#### sqlitesearch

sqlitesearch is the persistent sibling of minsearch, and it solves both problems at once.

We already used it in module 1 for persistent text search. It also does vector search through its **VectorSearchIndex** class. It stores vectors in SQLite, a real on-disk database, and uses ANN strategies for retrieval. Because the data lives on disk, one process can write the vectors and another can read them back.

If you didn't install it in the previous module, add it to your project:

uv add sqlitesearch

##### Creating the index

Initialize it:

In [30]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch supports three ANN modes:

* lsh (default): up to 100K vectors, random hyperplane projections
* ivf: 10K-500K vectors, K-means clustering
* hnsw: 10K-1M+ vectors, proximity graph (highest recall)
  
For our small dataset, lsh is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

##### Indexing the data

Fit the index with our vectors and documents:


In [31]:
vs_index.fit(vectors, documents)

The index is saved to faq_vectors2.db. 

Unlike minsearch, this file persists on disk. You can search immediately after indexing, or reopen the index later without re-indexing.

##### Searching

Search works the same way as with minsearch. We always encode the query into a vector first. 

This is one thing that makes vector search heavier than text search. With text search we'd throw the raw query straight at the engine.

Encode, then search:

In [32]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

Look at the results:

In [33]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

##### Filtering by course

Filtering works the same way:

In [34]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

##### Closing the connection

When you're done with the index:

In [35]:
vs_index.close()

##### Reopening the index

In a new Python session, you can reopen the index without re-computing embeddings:

In [36]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Now we can search:

In [37]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [38]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Module “kafka” not found when trying to run producer.py',
  'answer': 'To resolve the "Module \'kafka\' not found" error when running `producer.py`, you can create a virtual environment and install the required packages. Follow these steps:\n\n1. **Create a Virtual Environment**\n   \n   Run the following command to create a virtual environment:\n   \n   ```bash\n   python -m venv env\n   ```\n\n2. **Activate the Virtual Environment**\n\n   - On macOS and Linux:\n     \n     ```bash\n     source env/bin/activate\n     ```\n   \n   - On Windows:\n     \n     ```bash\n     env\\Scripts\\activate\n     ```\n\n3. **Install Required Packages**\n   \n   Install the packages listed in `requirements.txt`:\n   \n   ```bash\n   pip install -r ../requirements.txt\n   ```\n\n4. **Deactivate the Virtual Environment**\n   \n   When you\'re done, deactivate the virtual environment:\n   \n   ```bash\n   deactiv

We still load the embedding model to encode the query, but we don't re-embed all the documents. No fit call needed, because the index is already built and waiting on disk.

This is the same two-process split we used for text search in module 1. One process ingests and builds the index, another queries it.

It matters more here than with text search. Embedding the whole dataset takes about a minute. We don't want a user waiting that long when the app starts up. We pay that cost once during ingestion, and the query side starts up instantly.

#### Using sqlitesearch vector search in RAG

Let's use our persistent vector index in RAG.

In a new notebook, set up the model and open the index (same as the "Reopening the index" section above):

### Comparing minsearch and sqlitesearch for vector search

Here is how the two compare:

* **minsearch VectorSearch**: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
* **sqlitesearch VectorSearchIndex**: persistent (SQLite .db file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence

This is probably the last you'll hear of sqlitesearch. I built it for teaching, to show the ingestion-then-deployment split.

It does have a real use though. Its only dependencies are SQLite and numpy. So it runs on any host that offers a free SQLite database, where a dedicated vector database would cost extra. For most work you'll reach for something else, which is what we do next.